In [1]:
from pathlib import Path
import glob
import re

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from rdkit.Chem import rdMolTransforms
from rdkit.Chem import rdMolAlign

In [2]:
# for pdb_path in glob.glob("sampling/train_ref/*.pdb"):
#     pdb_num = int(re.search(r'(\d+).pdb', pdb_path).group(1))
#     mol = Chem.MolFromPDBFile(pdb_path)
#     Chem.MolToPDBFile(mol, f"sampling/test_{pdb_num}.pdb")

In [2]:
def load_pdb(path):
    mol = Chem.MolFromPDBFile(path, removeHs=True, sanitize=False)
    if mol is None:
        print(f"Failed to load {path}")
        return None
    for atom in mol.GetAtoms():
        atom.SetChiralTag(rdchem.ChiralType.CHI_UNSPECIFIED)

    for bond in mol.GetBonds():
        bond.SetStereo(rdchem.BondStereo.STEREONONE)
    try:
        Chem.SanitizeMol(mol)
    except:
        print(f"Failed to sanitize {path}")
        return None
    return mol

def heavy_atom_rmsd(mol_ref, mol_gen):
    # Remove hydrogens for heavy-atom RMSD
    ref = Chem.RemoveHs(mol_ref)
    gen = Chem.RemoveHs(mol_gen)

    return rdMolAlign.GetBestRMS(ref, gen)

def bond_length_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    errs = []
    for b in mol_ref.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()

        a_ri = mol_ref.GetAtomWithIdx(i)
        a_rj = mol_ref.GetAtomWithIdx(j)
        a_gi = mol_gen.GetAtomWithIdx(i)
        a_gj = mol_gen.GetAtomWithIdx(j)

        # atom identity checks
        assert mol_ref.GetNumAtoms() == mol_gen.GetNumAtoms()
        assert mol_ref.GetNumBonds() == mol_gen.GetNumBonds()

        for b in mol_ref.GetBonds():
            i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
            assert mol_gen.GetBondBetweenAtoms(i, j) is not None

        d_r = conf_r.GetAtomPosition(i).Distance(conf_r.GetAtomPosition(j))
        d_g = conf_g.GetAtomPosition(i).Distance(conf_g.GetAtomPosition(j))

        errs.append(abs(d_r - d_g))

    return float(np.mean(errs))


def angle(a, b, c):
    ba = a - b
    bc = c - b
    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)
    if norm_ba == 0 or norm_bc == 0:
        return 0.0  # Return 0 if vectors are zero (shouldn't happen in valid molecules)
    cos_angle = np.dot(ba, bc) / (norm_ba * norm_bc)
    # Clamp to [-1, 1] to avoid NaN from floating point precision errors
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))

def bond_angle_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    errs = []
    for atom in mol_ref.GetAtoms():
        nbrs = [n.GetIdx() for n in atom.GetNeighbors()]
        if len(nbrs) < 2:
            continue
        for i in range(len(nbrs)):
            for j in range(i + 1, len(nbrs)):
                a, b, c = nbrs[i], atom.GetIdx(), nbrs[j]
                ar = angle(conf_r.GetAtomPosition(a),
                           conf_r.GetAtomPosition(b),
                           conf_r.GetAtomPosition(c))
                ag = angle(conf_g.GetAtomPosition(a),
                           conf_g.GetAtomPosition(b),
                           conf_g.GetAtomPosition(c))
                errs.append(abs(ar - ag))
    return float(np.mean(errs))

def torsion_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    torsions = []

    for bond in mol_ref.GetBonds():
        if bond.IsInRing():
            continue
        if bond.GetBondType() != Chem.BondType.SINGLE:
            continue

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        ai = mol_ref.GetAtomWithIdx(i)
        aj = mol_ref.GetAtomWithIdx(j)

        if ai.GetDegree() < 2 or aj.GetDegree() < 2:
            continue

        ni = sorted(a.GetIdx() for a in ai.GetNeighbors() if a.GetIdx() != j)
        nj = sorted(a.GetIdx() for a in aj.GetNeighbors() if a.GetIdx() != i)

        for a in ni:
            for d in nj:
                tr = rdMolTransforms.GetDihedralDeg(conf_r, a, i, j, d)
                tg = rdMolTransforms.GetDihedralDeg(conf_g, a, i, j, d)

                diff = abs(tr - tg) % 360
                diff = min(diff, 360 - diff)
                torsions.append(diff)

    return float(np.mean(torsions)) if torsions else 0.0


def heavy_atom_rmsf(mol_list, align_first=True):
    """
    Calculate RMSF (Root Mean Square Fluctuation) for heavy atoms across multiple conformations.
    
    Args:
        mol_list: List of RDKit molecule objects (should be the same molecule, different conformations)
        align_first: If True, align all conformations to the first one before calculating RMSF
    
    Returns:
        Average RMSF over all heavy atoms
    """
    if len(mol_list) < 2:
        return 0.0
    
    # Remove hydrogens from all molecules
    mols_noH = [Chem.RemoveHs(mol) for mol in mol_list]
    
    # Check that all molecules have the same number of atoms
    n_atoms = mols_noH[0].GetNumAtoms()
    if not all(mol.GetNumAtoms() == n_atoms for mol in mols_noH):
        print("Warning: Not all conformations have the same number of atoms")
        return None
    
    # Get coordinates for all conformations
    coords_list = []
    for mol in mols_noH:
        conf = mol.GetConformer()
        coords = np.array([conf.GetAtomPosition(i) for i in range(n_atoms)])
        coords_list.append(coords)
    
    # Stack coordinates: [n_confs, n_atoms, 3]
    coords_array = np.stack(coords_list, axis=0)
    
    # Optionally align all conformations to the first one
    if align_first:
        ref_coords = coords_array[0]
        aligned_coords = [ref_coords]  # First conformation is reference
        for i in range(1, coords_array.shape[0]):
            # Align conformation i to reference using Kabsch algorithm
            mol_ref = mols_noH[0]
            mol_to_align = mols_noH[i]
            rmsd = rdMolAlign.GetBestRMS(mol_ref, mol_to_align)
            # Get the aligned conformation
            mol_aligned = Chem.Mol(mol_to_align)
            rdMolAlign.AlignMol(mol_aligned, mol_ref)
            conf_aligned = mol_aligned.GetConformer()
            coords_aligned = np.array([conf_aligned.GetAtomPosition(j) for j in range(n_atoms)])
            aligned_coords.append(coords_aligned)
        coords_array = np.stack(aligned_coords, axis=0)
    
    # Calculate mean position for each atom across conformations
    mean_coords = np.mean(coords_array, axis=0)  # [n_atoms, 3]
    
    # Calculate RMSF for each atom: sqrt(mean((pos_i - mean_pos)^2))
    squared_deviations = np.sum((coords_array - mean_coords[None, :, :]) ** 2, axis=2)  # [n_confs, n_atoms]
    rmsf_per_atom = np.sqrt(np.mean(squared_deviations, axis=0))  # [n_atoms]
    
    # Return average RMSF over all heavy atoms
    return float(np.mean(rmsf_per_atom))



def mmff_energy(mol):
    # AllChem.MMFFOptimizeMolecule(mol)
    props = AllChem.MMFFGetMoleculeProperties(mol)
    ff = AllChem.MMFFGetMoleculeForceField(mol, props)
    return ff.CalcEnergy()


from rdkit.Chem import rdchem
from rdkit.Chem import rdmolops


def clash_count(mol, scale=0.75):
    conf = mol.GetConformer()
    pts = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])
    vdw = np.array([
        rdchem.GetPeriodicTable().GetRvdw(a.GetAtomicNum())
        for a in mol.GetAtoms()
    ])

    n = mol.GetNumAtoms()
    clashes = 0

    for i in range(n):
        if mol.GetAtomWithIdx(i).GetAtomicNum() == 1:
            continue
        for j in range(i + 1, n):
            if mol.GetAtomWithIdx(j).GetAtomicNum() == 1:
                continue

            path = rdmolops.GetShortestPath(mol, i, j)
            if path is not None and len(path) <= 4:
                continue

            if np.linalg.norm(pts[i] - pts[j]) < scale * (vdw[i] + vdw[j]):
                clashes += 1

    return clashes

In [3]:
import glob
from pathlib import Path
from collections import defaultdict
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdMolAlign, rdmolfiles
from joblib import Parallel, delayed
import multiprocessing as mp

# ============================================================
# USER SETTINGS
# ============================================================

mode = "train"

REF_GLOB = f"sampling/geom_drugs_murcko_{mode}_ref/*.pdb"
PRED_GLOB = f"sampling/geom_identityRot_256_conformer_{mode}_3std_bondlength/samples/*.pdb"

OUT_ALIGN_DIR = Path(f"aligned_pairs/geom_identityRot_256_conformer/{mode}")   # where aligned PDBs are saved

N_JOBS = max(1, mp.cpu_count() // 2)
USE_PARALLEL = True

OUT_ALIGN_DIR.mkdir(exist_ok=True, parents=True)

# ============================================================
# REQUIRED USER FUNCTIONS (already in your codebase)
# ============================================================

# load_pdb(path)
# heavy_atom_rmsd(mol1, mol2)
# heavy_atom_rmsf(mol_list, align_first=True)
# bond_length_mae(mol1, mol2)
# bond_angle_mae(mol1, mol2)
# torsion_mae(mol1, mol2)

# ============================================================
# Utilities
# ============================================================

def canon_smi_from_mol(mol):
    return Chem.CanonSmiles(Chem.MolToSmiles(mol))


def load_and_prepare(pdb_path):
    mol = load_pdb(pdb_path)
    if mol is None:
        return None, None
    mol = Chem.AddHs(mol)
    smi = canon_smi_from_mol(mol)
    return mol, smi


def align_copy(mol_probe, mol_ref):
    """Return aligned COPIES (does not modify originals)"""
    probe = Chem.Mol(mol_probe)
    ref = Chem.Mol(mol_ref)
    rdMolAlign.AlignMol(probe, ref)
    return probe, ref


from rdkit.Chem import rdmolfiles

def pdb_block_full_conect(mol):
    # flavor=4 → write CONECT in both directions (required by many viewers)
    return rdmolfiles.MolToPDBBlock(mol, flavor=4)

def save_best_pair(m1, m2, out_path):
    m1 = Chem.RemoveHs(m1)
    m2 = Chem.RemoveHs(m2)

    with open(out_path, "w") as f:
        f.write("MODEL        1\n")
        f.write(rdmolfiles.MolToPDBBlock(m1, flavor=4))  # full CONECT
        f.write("ENDMDL\n")

        f.write("MODEL        2\n")
        f.write(rdmolfiles.MolToPDBBlock(m2, flavor=4))
        f.write("ENDMDL\n")

        f.write("END\n")



# ============================================================
# Load Reference Molecules
# ============================================================

print("Loading reference molecules...")

ref_smi_to_mols = defaultdict(list)

for ref_path in glob.glob(REF_GLOB):
    mol, smi = load_and_prepare(ref_path)
    assert mol is not None, f"ref mol is None: {ref_path}"
    ref_smi_to_mols[smi].append(mol)

print(f"Loaded {sum(len(v) for v in ref_smi_to_mols.values())} reference conformers")
print(f"{len(ref_smi_to_mols)} unique SMILES in reference")


# ============================================================
# Collect Predicted Molecules
# ============================================================

print("Collecting predicted PDBs...")

pred_paths = [Path(p) for p in glob.glob(PRED_GLOB)]

groups = defaultdict(list)
for p in pred_paths:
    mol_id = p.stem.rsplit("_", 1)[0]
    groups[mol_id].append(p)

print(f"{len(groups)} molecule groups found")


# ============================================================
# Metric + Save ALL aligned candidates
# ============================================================

def compute_metrics_for_group(paths):

    mol_confs = []
    best_metrics = None
    valid = 0

    for p in sorted(paths):

        pred_mol, smi = load_and_prepare(p)
        if pred_mol is None:
            continue

        ref_mols = ref_smi_to_mols.get(smi)
        if ref_mols is None:
            continue

        best_idx = None
        best_rmsd = float("inf")
        best_vals = None

        # ---- Find best ref ----
        for ref_idx, ref in enumerate(ref_mols):
            r = heavy_atom_rmsd(pred_mol, ref)
            if r < best_rmsd:
                best_rmsd = r
                best_idx = ref_idx
                best_vals = (
                    r,
                    bond_length_mae(pred_mol, ref),
                    bond_angle_mae(pred_mol, ref),
                    torsion_mae(pred_mol, ref),
                    clash_count(pred_mol),
                )

        if best_idx is None:
            continue

        best_ref = ref_mols[best_idx]

        # ---- Align + save ONLY best ----
        probe_aligned = Chem.Mol(pred_mol)
        ref_aligned = Chem.Mol(best_ref)
        rdMolAlign.AlignMol(probe_aligned, ref_aligned)

        out_name = f"{p.stem}_BEST_rmsd{best_rmsd:.3f}.pdb"
        out_path = OUT_ALIGN_DIR / out_name
        save_best_pair(probe_aligned, ref_aligned, out_path)

        mol_confs.append(pred_mol)
        valid += 1

        if best_metrics is None or best_vals[0] < best_metrics[0]:
            best_metrics = best_vals

    # ---- RMSF across conformers ----
    rmsf_val = None
    if len(mol_confs) >= 2:
        rmsf_val = heavy_atom_rmsf(mol_confs, align_first=True)

    return best_metrics, rmsf_val, valid


# ============================================================
# Run Computation
# ============================================================

print("Running metric computation...")

if USE_PARALLEL:
    results = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(compute_metrics_for_group)(paths)
        for paths in groups.values()
    )
else:
    results = [compute_metrics_for_group(paths) for paths in groups.values()]


# ============================================================
# Aggregate Results
# ============================================================

heavy_atom_min_rmsd = []
bond_length_min_mae = []
bond_angle_min_mae = []
torsion_min_mae = []
clash_list = []
rmsf_list = []

valid_mols = 0

for best_metrics, rmsf_val, valid in results:

    valid_mols += valid

    if best_metrics is not None:
        heavy_atom_min_rmsd.append(best_metrics[0])
        bond_length_min_mae.append(best_metrics[1])
        bond_angle_min_mae.append(best_metrics[2])
        torsion_min_mae.append(best_metrics[3])
        clash_list.append(best_metrics[4])

    if rmsf_val is not None:
        rmsf_list.append(rmsf_val)


# ============================================================
# Summary
# ============================================================

print("\n===== SUMMARY =====")
print("Valid molecules:", valid_mols)
print("Num RMSD entries:", len(heavy_atom_min_rmsd))
print("Num RMSF entries:", len(rmsf_list))

print("Mean Heavy Atom RMSD:", np.mean(heavy_atom_min_rmsd))
print("Mean Bond Length MAE:", np.mean(bond_length_min_mae))
print("Mean Bond Angle MAE:", np.mean(bond_angle_min_mae))
print("Mean Torsion MAE:", np.mean(torsion_min_mae))
print("Mean Clashes:", np.mean(clash_list))
print("Mean RMSF:", np.mean(rmsf_list))

Loading reference molecules...
Loaded 10767 reference conformers
60 unique SMILES in reference
30 molecule groups found
Running metric computation...

===== SUMMARY =====
Valid molecules: 120
Num RMSD entries: 24
Num RMSF entries: 24
Mean Heavy Atom RMSD: 1.1195024016908812
Mean Bond Length MAE: 0.6525118254553836
Mean Bond Angle MAE: 13.644594497221737
Mean Torsion MAE: 51.12176283088086
Mean Clashes: 0.0
Mean RMSF: 3.1807775791799933


In [10]:
import glob
from pathlib import Path
from collections import defaultdict
import numpy as np
from rdkit import Chem
from joblib import Parallel, delayed
import multiprocessing as mp

# ============================================================
# USER SETTINGS
# ============================================================

mode = "test"

REF_GLOB = f"sampling/geom_drugs_murcko_{mode}_ref/*.pdb"
PRED_GLOB = f"sampling/geom_identityRot_256_conformer_{mode}_3std_bondlength/samples/*.pdb"

DELTA = 0.75   # RMSD threshold for coverage (Å)

N_JOBS = max(1, mp.cpu_count() // 2)
USE_PARALLEL = True


# ============================================================
# REQUIRED USER FUNCTIONS (must already exist in your codebase)
# ============================================================

# load_pdb(path)
# heavy_atom_rmsd(mol1, mol2)
# heavy_atom_rmsf(mol_list, align_first=True)
# bond_length_mae(mol1, mol2)
# bond_angle_mae(mol1, mol2)
# torsion_mae(mol1, mol2)


# ============================================================
# Utilities
# ============================================================

def canon_smi_from_mol(mol):
    return Chem.CanonSmiles(Chem.MolToSmiles(mol))


def load_and_prepare(pdb_path):
    mol = load_pdb(pdb_path)
    if mol is None:
        return None, None
    mol = Chem.AddHs(mol)
    smi = canon_smi_from_mol(mol)
    return mol, smi


def pairwise_rmsd_matrix(ref_mols, gen_mols):
    n_r = len(ref_mols)
    n_g = len(gen_mols)
    D = np.zeros((n_r, n_g), dtype=float)

    for i, r in enumerate(ref_mols):
        for j, g in enumerate(gen_mols):
            D[i, j] = heavy_atom_rmsd(r, g)

    return D

def compute_reference_rmsf_mean(ref_smi_to_mols):
    vals = []
    for mols in ref_smi_to_mols.values():
        if len(mols) < 2:
            continue
        try:
            v = heavy_atom_rmsf(mols, align_first=True)
            if v is not None and not np.isnan(v):
                vals.append(v)
        except:
            pass
    return float(np.mean(vals)) if len(vals) > 0 else None


# ============================================================
# Load Reference Conformers
# ============================================================

print("Loading reference molecules...")

ref_smi_to_mols = defaultdict(list)
ref_mol_id_to_smiles = {}  # mol_id (from ref path stem) -> ref canonical SMILES

for ref_path in glob.glob(REF_GLOB):
    mol, smi = load_and_prepare(ref_path)
    assert mol is not None, f"Failed to load ref: {ref_path}"
    ref_smi_to_mols[smi].append(mol)
    mol_id = Path(ref_path).stem.rsplit("_", 1)[0]
    ref_mol_id_to_smiles[mol_id] = smi

print(f"Loaded {sum(len(v) for v in ref_smi_to_mols.values())} reference conformers")
print(f"{len(ref_smi_to_mols)} unique molecules")


# ============================================================
# Gather Predicted Conformers
# ============================================================

print("Collecting predicted molecules...")

pred_paths = [Path(p) for p in glob.glob(PRED_GLOB)]

groups = defaultdict(list)
for p in pred_paths:
    mol_id = p.stem.rsplit("_", 1)[0]  # same convention as ref: base name before _N
    groups[mol_id].append(p)

print(f"{len(groups)} molecule groups found")


# ============================================================
# Metric Computation Per Molecule
# ============================================================

def compute_metrics_for_group(paths, ref_smi_to_mols):

    gen_mols = []
    ref_mols_for_group = None
    no_match_pairs = []  # (mol_id, gen_smiles) when gen SMILES not in ref_smi_to_mols; ref_smi resolved in main process
    mol_id = paths[0].stem.rsplit("_", 1)[0] if paths else None

    # ------------------------------
    # Load generated conformers
    # ------------------------------
    for p in sorted(paths):
        pred_mol, smi = load_and_prepare(p)
        if pred_mol is None:
            continue

        ref_mols = ref_smi_to_mols.get(smi)
        if ref_mols is None:
            no_match_pairs.append((mol_id, smi))
            continue

        gen_mols.append(pred_mol)
        ref_mols_for_group = ref_mols

    if not gen_mols or ref_mols_for_group is None:
        return (None, no_match_pairs)

    # ============================================================
    # Pairwise RMSD Matrix
    # ============================================================
    D = pairwise_rmsd_matrix(ref_mols_for_group, gen_mols)

    # ------------------------------
    # AMR-R / Coverage-R (Recall)
    # ------------------------------
    min_ref = D.min(axis=1)     # per reference conformer
    amr_r = float(min_ref.mean())
    cov_r = float((min_ref < DELTA).mean())

    # ------------------------------
    # AMR-P / Coverage-P (Precision)
    # ------------------------------
    min_gen = D.min(axis=0)     # per generated conformer
    amr_p = float(min_gen.mean())
    cov_p = float((min_gen < DELTA).mean())

    # ============================================================
    # Geometry Metrics (Best match per generated conformer)
    # ============================================================
    best_rmsd = []
    best_bl = []
    best_ba = []
    best_tor = []

    for j, gen in enumerate(gen_mols):
        i = int(np.argmin(D[:, j]))
        ref = ref_mols_for_group[i]

        best_rmsd.append(D[i, j])
        best_bl.append(bond_length_mae(gen, ref))
        best_ba.append(bond_angle_mae(gen, ref))
        best_tor.append(torsion_mae(gen, ref))

    # ============================================================
    # RMSF Across Generated Conformers
    # ============================================================
    rmsf_val = None
    if len(gen_mols) >= 2:
        rmsf_val = heavy_atom_rmsf(gen_mols, align_first=True)

    return (
        (
            amr_r,
            cov_r,
            amr_p,
            cov_p,
            np.mean(best_rmsd),
            np.mean(best_bl),
            np.mean(best_ba),
            np.mean(best_tor),
            rmsf_val,
        ),
        no_match_pairs,
    )


# ============================================================
# Run Computation
# ============================================================

print("Running metric computation...")

if USE_PARALLEL:
    results = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(compute_metrics_for_group)(paths, ref_smi_to_mols)
        for paths in groups.values()
    )
else:
    results = [
        compute_metrics_for_group(paths, ref_smi_to_mols)
        for paths in groups.values()
    ]


# ============================================================
# Aggregate Results
# ============================================================

amr_r_list = []
cov_r_list = []
amr_p_list = []
cov_p_list = []
rmsd_list = []
bl_list = []
ba_list = []
tor_list = []
rmsf_list = []
no_match_all = []  # (mol_id, gen_smiles) from workers; ref_smi resolved below in main process

for res in results:
    metrics, no_match_pairs = res if isinstance(res, tuple) else (res, [])
    if no_match_pairs:
        no_match_all.extend(no_match_pairs)
    if metrics is None:
        continue

    (
        amr_r,
        cov_r,
        amr_p,
        cov_p,
        rmsd,
        bl,
        ba,
        tor,
        rmsf_val,
    ) = metrics

    amr_r_list.append(amr_r)
    cov_r_list.append(cov_r)
    amr_p_list.append(amr_p)
    cov_p_list.append(cov_p)

    rmsd_list.append(rmsd)
    bl_list.append(bl)
    ba_list.append(ba)
    tor_list.append(tor)

    if rmsf_val is not None:
        rmsf_list.append(rmsf_val)

# Resolve ref_smi from ref_mol_id_to_smiles (already built when we loaded refs in this cell)
no_match_resolved = []
for mol_id, gen_smi in no_match_all:
    ref_smi = ref_mol_id_to_smiles.get(mol_id) if mol_id else None
    no_match_resolved.append((ref_smi, gen_smi))

ref_rmsf_mean = compute_reference_rmsf_mean(ref_smi_to_mols)

# ============================================================
# Summary
# ============================================================

print("\n==============================")
print("        FINAL RESULTS")
print("==============================")

print("\n--- Coverage / Precision ---")
print("AMR-R (Recall Quality):", np.mean(amr_r_list))
print("COV-R (Recall Coverage):", np.mean(cov_r_list))
print("AMR-P (Precision Quality):", np.mean(amr_p_list))
print("COV-P (Precision Coverage):", np.mean(cov_p_list))

print("\n--- Geometry ---")
print("Mean Heavy Atom RMSD:", np.mean(rmsd_list))
print("Mean Bond Length MAE:", np.mean(bl_list))
print("Mean Bond Angle MAE:", np.mean(ba_list))
print("Mean Torsion MAE:", np.mean(tor_list))

print("\n--- Diversity ---")
print("Mean RMSF:", np.mean(rmsf_list))

print("Reference RMSF mean:", ref_rmsf_mean)

if no_match_resolved:
    print("\n--- No-match (ref SMILES vs generated SMILES) ---")
    seen = set()
    for ref_smi, gen_smi in no_match_resolved:
        key = (ref_smi, gen_smi)
        if key not in seen:
            seen.add(key)
            print("  ref:", ref_smi)
            print("  gen:", gen_smi)
            print()
else:
    print("\n--- No-match: none (all generated SMILES had a ref) ---")

Loading reference molecules...
Loaded 4531 reference conformers
30 unique molecules
59 molecule groups found
Running metric computation...


[18:03:26] Explicit valence for atom # 19 Br, 2, is greater than permitted
[18:03:26] Explicit valence for atom # 19 Br, 2, is greater than permitted


Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/abdf6df45e6ec7547d205fd964ce976d18a2b0ad6e0704b6f05c4225beade605_1.pdb
Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/abdf6df45e6ec7547d205fd964ce976d18a2b0ad6e0704b6f05c4225beade605_3.pdb


[18:03:26] Explicit valence for atom # 0 Br, 2, is greater than permitted
[18:03:26] Explicit valence for atom # 0 Br, 2, is greater than permitted
[18:03:26] Explicit valence for atom # 0 Br, 2, is greater than permitted
[18:03:26] Explicit valence for atom # 0 Br, 3, is greater than permitted
[18:03:26] Explicit valence for atom # 0 Br, 2, is greater than permitted


Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/a30ffe4265341dec80fc4d4834da21120a7e3c60427feedd18dd1fff8ecadd90_0.pdb
Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/a30ffe4265341dec80fc4d4834da21120a7e3c60427feedd18dd1fff8ecadd90_1.pdb
Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/a30ffe4265341dec80fc4d4834da21120a7e3c60427feedd18dd1fff8ecadd90_2.pdb
Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/a30ffe4265341dec80fc4d4834da21120a7e3c60427feedd18dd1fff8ecadd90_3.pdb
Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/a30ffe4265341dec80fc4d4834da21120a7e3c60427feedd18dd1fff8ecadd90_4.pdb


...

Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/0ad5197f2dd08954f4d691e5349781554f12236e4fc352867bbb8887561d0348_2.pdb


[18:03:45] Explicit valence for atom # 0 Cl, 2, is greater than permitted


Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/7cb91fccb7ce664ae6d8b82597565613bcdfe740bffb49d91596dba5621137ca_0.pdb


[18:03:46] Explicit valence for atom # 0 Cl, 5, is greater than permitted
[18:03:46] Explicit valence for atom # 0 Cl, 3, is greater than permitted
[18:03:46] Explicit valence for atom # 0 Cl, 2, is greater than permitted


Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/1165b7e80cbc58f4eb5e74bcbd33b2f05f089ffc3ff5bd843d8a0eb7e69ff993_0.pdb
Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/1165b7e80cbc58f4eb5e74bcbd33b2f05f089ffc3ff5bd843d8a0eb7e69ff993_1.pdb
Failed to sanitize sampling/geom_identityRot_256_conformer_test_3std_bondlength/samples/1165b7e80cbc58f4eb5e74bcbd33b2f05f089ffc3ff5bd843d8a0eb7e69ff993_4.pdb

        FINAL RESULTS

--- Coverage / Precision ---
AMR-R (Recall Quality): 1.5347906779842193
COV-R (Recall Coverage): 0.05142373285669229
AMR-P (Precision Quality): 1.3167012132746219
COV-P (Precision Coverage): 0.17592592592592593

--- Geometry ---
Mean Heavy Atom RMSD: 1.3167012132746219
Mean Bond Length MAE: 0.8680137808964495
Mean Bond Angle MAE: 16.578542273248235
Mean Torsion MAE: 57.224999215137

--- Diversity ---
Mean RMSF: 2.83053657315706
Reference RMSF mean: 2.956360262204351

--- No-match (ref SMILES vs

In [7]:
# train - all
print(np.mean(heavy_atom_min_rmsd_list), len(heavy_atom_min_rmsd_list))
print(np.mean(bond_length_min_mae_list))
print(np.mean(bond_angle_min_mae_list))
print(np.mean(torsion_min_mae_list))
print(np.mean(rmsf_list), len(rmsf_list))
print(valid_mols)
# print(np.mean(energy_list), len(energy_list))

0.9267492452212903 29
0.545315313830162
15.708515700056578
61.18656122378276
2.7537358466653465 29
142


In [9]:
print(np.mean(heavy_atom_min_rmsd_list), len(heavy_atom_min_rmsd_list))
print(np.mean(bond_length_min_mae_list))
print(np.mean(bond_angle_min_mae_list))
print(np.mean(torsion_min_mae_list))
print(np.mean(rmsf_list), len(rmsf_list))
print(valid_mols)

nan 0
nan
nan
nan
nan 0
0


/var/local/dy4652/miniconda3/envs/proteinzen/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/var/local/dy4652/miniconda3/envs/proteinzen/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
